# Capstone Project 1: Retail Sales Analysis & Insights

**Business Context**  
You are a data analyst at “UrbanCart”, a multi-category retail company.  
Management wants a clear picture of last year’s performance and actionable recommendations.

**Dataset**  
We will generate a realistic synthetic sales dataset (you can replace it with any real CSV later).

**Deliverables**
1. Clean and explore the data
2. Key KPIs (Revenue, Orders, AOV, Growth)
3. Regional & Category performance
4. Customer segmentation (simple RFM-style)
5. Short executive summary with 3 recommendations


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

np.random.seed(42)
sns.set_style("whitegrid")

# Generate synthetic data
n = 5000
start_date = datetime(2024, 1, 1)
dates = [start_date + timedelta(days=np.random.randint(0, 365)) for _ in range(n)]

df = pd.DataFrame({
    "order_id": range(10001, 10001 + n),
    "order_date": dates,
    "region": np.random.choice(["North", "South", "East", "West"], n, p=[0.3, 0.25, 0.25, 0.2]),
    "category": np.random.choice(["Electronics", "Apparel", "Home", "Grocery", "Beauty"], n),
    "quantity": np.random.randint(1, 6, n),
    "unit_price": np.round(np.random.uniform(100, 5000, n), 0),
    "customer_id": np.random.randint(1000, 2000, n),
    "channel": np.random.choice(["Online", "Store", "App"], n, p=[0.5, 0.3, 0.2])
})
df["revenue"] = df["quantity"] * df["unit_price"]
df["order_date"] = pd.to_datetime(df["order_date"])
df["month"] = df["order_date"].dt.to_period("M")

print(df.head())
print("\nShape:", df.shape)
print("\nDate range:", df["order_date"].min().date(), "→", df["order_date"].max().date())


In [ ]:
# 1. High-level KPIs
total_revenue = df["revenue"].sum()
total_orders = df["order_id"].nunique()
aov = total_revenue / total_orders
unique_customers = df["customer_id"].nunique()

print("="*40)
print("URBANCART – ANNUAL SNAPSHOT")
print("="*40)
print(f"Total Revenue      : ₹{total_revenue:,.0f}")
print(f"Total Orders       : {total_orders:,}")
print(f"Average Order Value: ₹{aov:,.0f}")
print(f"Unique Customers   : {unique_customers:,}")


In [ ]:
# 2. Monthly trend
monthly = df.groupby("month")["revenue"].sum()
monthly.index = monthly.index.astype(str)

plt.figure(figsize=(12, 5))
monthly.plot(kind="line", marker="o")
plt.title("Monthly Revenue Trend")
plt.ylabel("Revenue (₹)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# 3. Region & Category performance
region_rev = df.groupby("region")["revenue"].sum().sort_values(ascending=False)
cat_rev = df.groupby("category")["revenue"].sum().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
region_rev.plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("Revenue by Region")
axes[0].set_ylabel("Revenue (₹)")

cat_rev.plot(kind="bar", ax=axes[1], color="darkorange")
axes[1].set_title("Revenue by Category")
plt.tight_layout()
plt.show()

print("Top Region:", region_rev.index[0], f"(₹{region_rev.iloc[0]:,.0f})")
print("Top Category:", cat_rev.index[0], f"(₹{cat_rev.iloc[0]:,.0f})")


In [ ]:
# 4. Simple RFM-style segmentation
snapshot = df["order_date"].max() + timedelta(days=1)
rfm = df.groupby("customer_id").agg(
    recency=("order_date", lambda x: (snapshot - x.max()).days),
    frequency=("order_id", "count"),
    monetary=("revenue", "sum")
).reset_index()

# Score 1-4
rfm["R_score"] = pd.qcut(rfm["recency"], 4, labels=[4,3,2,1])  # lower recency better
rfm["F_score"] = pd.qcut(rfm["frequency"].rank(method="first"), 4, labels=[1,2,3,4])
rfm["M_score"] = pd.qcut(rfm["monetary"], 4, labels=[1,2,3,4])
rfm["RFM_Score"] = rfm["R_score"].astype(int) + rfm["F_score"].astype(int) + rfm["M_score"].astype(int)

def segment(score):
    if score >= 10: return "Champions"
    elif score >= 8: return "Loyal"
    elif score >= 6: return "Potential"
    else: return "At Risk"

rfm["Segment"] = rfm["RFM_Score"].apply(segment)
print(rfm["Segment"].value_counts())
print("\nAverage monetary by segment:")
print(rfm.groupby("Segment")["monetary"].mean().round(0))


---
## Executive Summary Template (fill after running the analysis)

**UrbanCart FY Performance**

1. **Overall**: Total revenue was ₹X with Y orders and an AOV of ₹Z.
2. **Geographic**: North region contributed the highest share (A%). West is under-performing.
3. **Category**: Electronics remains the engine; Beauty has high growth potential.
4. **Customers**: Champions (top RFM) generate disproportionate revenue – protect them with loyalty programmes.
5. **Recommendations**:
   - Increase marketing budget in the highest-ROI region/category.
   - Launch a win-back campaign for “At Risk” customers.
   - Test higher AOV tactics (bundles) in the App channel.
